In [ ]:

import pandas as pd
from pathlib import Path

print("Pandas version:", pd.__version__)
print("Current folder:", Path.cwd()) 


In [ ]:

# Step 1: Define the raw data folder
data_path = Path("../Data/Raw/archive")

# Step 2: Load the customers dataset
customers = pd.read_csv(
    data_path / "olist_customers_dataset.csv"
)

# Step 3: Preview the first 5 rows
customers.head()


In [ ]:

print("Dataset shape:", customers.shape)

print("\nMissing values:")
print(customers.isnull().sum())

print("\nDuplicate rows:")
print(customers.duplicated().sum())

print("\nUnique customers:")
print(customers["customer_unique_id"].nunique())


In [ ]:

# Load orders and order items
orders = pd.read_csv(
    data_path / "olist_orders_dataset.csv"
)

order_items = pd.read_csv(
    data_path / "olist_order_items_dataset.csv"
)

print("Orders shape:", orders.shape)
print("Order items shape:", order_items.shape)

display(orders.head())
display(order_items.head())


In [ ]:

# Check data quality
for name, df in [
    ("Orders", orders),
    ("Order Items", order_items)
]:
    print(f"\n--- {name} ---")
    print("Rows:", len(df))
    print("Duplicate rows:", df.duplicated().sum())
    print("\nMissing values:")
    print(df.isnull().sum())


In [ ]:

# Count orders by status
status_counts = orders["order_status"].value_counts()

print("Order status distribution:")
print(status_counts)

# Missing delivery dates by order status
print("\nMissing customer delivery dates by status:")
print(
    orders.groupby("order_status")[
        "order_delivered_customer_date"
    ].apply(lambda x: x.isna().sum())
)


In [ ]:

print("Unique order IDs in orders:",
      orders["order_id"].nunique())

print("Unique order IDs in order items:",
      order_items["order_id"].nunique())

print("Orders without item records:",
      (~orders["order_id"].isin(
          order_items["order_id"]
      )).sum())

print("Items without matching orders:",
      (~order_items["order_id"].isin(
          orders["order_id"]
      )).sum())


In [ ]:

# Identify orders without item records
orders_without_items = orders[
    ~orders["order_id"].isin(order_items["order_id"])
]

print("Orders without items by status:")
print(
    orders_without_items["order_status"].value_counts()
)

# Check delivered orders without delivery dates
print("\nDelivered orders missing delivery date:")
print(
    orders[
        (orders["order_status"] == "delivered") &
        (orders["order_delivered_customer_date"].isna())
    ].shape[0]
)

# Check data types of item prices
print("\nPrice and freight data types:")
print(order_items[["price", "freight_value"]].dtypes)


In [ ]:

# 1. Keep delivered orders for sales analysis
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

# 2. Convert purchase timestamps to datetime
delivered_orders["order_purchase_timestamp"] = pd.to_datetime(
    delivered_orders["order_purchase_timestamp"]
)

# 3. Join orders with customers
sales = delivered_orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# 4. Join with order items
sales = sales.merge(
    order_items,
    on="order_id",
    how="inner",
    validate="one_to_many"
)

# 5. Check results
print("Sales rows:", len(sales))
print("Unique orders:", sales["order_id"].nunique())
print("Missing customer IDs:", sales["customer_unique_id"].isna().sum())

display(sales.head())


In [ ]:

# Calculate core sales KPIs
total_revenue = sales["price"].sum()
total_orders = sales["order_id"].nunique()
average_order_value = total_revenue / total_orders

print(f"Product Revenue: R$ {total_revenue:,.2f}")
print(f"Delivered Orders: {total_orders:,}")
print(f"Average Order Value: R$ {average_order_value:,.2f}")


In [ ]:

# Extract purchase month
sales["purchase_month"] = (
    sales["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

# Calculate monthly sales KPIs
monthly_sales = (
    sales.groupby("purchase_month")
    .agg(
        revenue=("price", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
)

monthly_sales["aov"] = (
    monthly_sales["revenue"] / monthly_sales["orders"]
)

display(monthly_sales)


In [ ]:

# Count distinct delivered orders per customer
customer_orders = (
    sales.groupby("customer_unique_id")["order_id"]
    .nunique()
)

total_customers = len(customer_orders)

repeat_customers = (customer_orders > 1).sum()

repeat_purchase_rate = (
    repeat_customers / total_customers * 100
)

print(f"Total Customers: {total_customers:,}")
print(f"Repeat Customers: {repeat_customers:,}")
print(f"Repeat Purchase Rate: {repeat_purchase_rate:.2f}%")


In [ ]:

# Create output folder if needed
cleaned_path = Path("../Data/Cleaned")
cleaned_path.mkdir(parents=True, exist_ok=True)

# Export analysis-ready datasets
sales.to_csv(
    cleaned_path / "sales_cleaned.csv",
    index=False
)

monthly_sales.to_csv(
    cleaned_path / "monthly_sales.csv",
    index=False
)

print("Files exported successfully!")


In [ ]:

import importlib.util

print(
    "SQLAlchemy installed:",
    importlib.util.find_spec("sqlalchemy") is not None
)

print(
    "PyMySQL installed:",
    importlib.util.find_spec("pymysql") is not None
)


In [ ]:

%pip install pymysql


In [ ]:

import pymysql

print("PyMySQL is ready!")


In [ ]:

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

# Enter your MySQL password
password = getpass("MySQL root password: ")

# Configure database connection
db_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password=password,
    host="127.0.0.1",
    port=3306,
    database="retail_analytics"
)

engine = create_engine(db_url)

# Test database connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT DATABASE()"))
    print("Connected to:", result.scalar())


In [ ]:

print("Rows ready for import:", len(sales))
print("Columns:", len(sales.columns))
print("Missing prices:", sales["price"].isna().sum())


In [ ]:

# Import the full sales dataset into a new MySQL table
sales.to_sql(
    name="sales_cleaned_v2",
    con=engine,
    if_exists="fail",
    index=False,
    chunksize=1000,
    method="multi"
)

print("Import completed successfully!")


In [ ]:

with engine.connect() as conn:
    result = conn.execute(
        text("SELECT COUNT(*) FROM sales_cleaned_v2")
    )
    print("MySQL imported rows:", result.scalar())
